In [1]:
# Import necessary libraries

import asyncio
import io
import os
from tkinter import Tk, filedialog
import edge_tts
import pandas as pd
import pygame
from tqdm.notebook import tqdm

pygame 2.6.1 (SDL 2.28.4, Python 3.11.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
# Fetch and display the list of available voices
async def get_comprehensive_voice_table():
    voices_manager = await edge_tts.VoicesManager.create()
    voices = voices_manager.voices
    
    processed_voices = []
    for v in voices:
        # Safely extract the nested VoiceTag data if it exists
        voice_tag = v.get("VoiceTag", {})
        categories = voice_tag.get("ContentCategories", [])
        personalities = voice_tag.get("VoicePersonalities", [])
        
        processed_voices.append({
            "ShortName": v.get("ShortName"),
            "Gender": v.get("Gender"),
            "Personalities": ", ".join(personalities) if personalities else "None",
            "Locale": v.get("Locale"),
            "Language": v.get("Language"),
            "FriendlyName": v.get("FriendlyName"),
            "SuggestedCodec": v.get("SuggestedCodec"),
            "Categories": ", ".join(categories) if categories else "None"
        })
    
    return pd.DataFrame(processed_voices)

# Generate the master DataFrame table variable
voice_df = await get_comprehensive_voice_table()

# # To see all rows, uncomment the line below
# voice_df

# # Display the first 5 rows of the interactive table in Jupyter
# voice_df.head(5)

# Filter the table to only include rows where the Language is English
english_voices_df = voice_df[voice_df['Language'] == 'en']

# Display the filtered table
english_voices_df.head(5)

,ShortName,Gender,Personalities,Locale,Language,FriendlyName,SuggestedCodec,Categories
77,en-AU-WilliamMultilingualNeural,Male,"Friendly, Positive",en-AU,en,Microsoft WilliamMultilingual Online (Natural)...,audio-24khz-48kbitrate-mono-mp3,General
78,en-AU-NatashaNeural,Female,"Friendly, Positive",en-AU,en,Microsoft Natasha Online (Natural) - English (...,audio-24khz-48kbitrate-mono-mp3,General
79,en-CA-ClaraNeural,Female,"Friendly, Positive",en-CA,en,Microsoft Clara Online (Natural) - English (Ca...,audio-24khz-48kbitrate-mono-mp3,General
80,en-CA-LiamNeural,Male,"Friendly, Positive",en-CA,en,Microsoft Liam Online (Natural) - English (Can...,audio-24khz-48kbitrate-mono-mp3,General
81,en-HK-YanNeural,Female,"Friendly, Positive",en-HK,en,Microsoft Yan Online (Natural) - English (Hong...,audio-24khz-48kbitrate-mono-mp3,General


In [4]:
# Define the text-to-speech function that uses edge-tts to convert text to 
#   speech and returns an in-memory audio buffer

async def text_to_speech(text, rate="0%", voice="en-US-AnaNeural"):
    """Streams TTS audio into an in-memory buffer."""
    communicate = edge_tts.Communicate(text=text, rate=rate, voice=voice)
    audio_buffer = io.BytesIO()
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            audio_buffer.write(chunk["data"])
    audio_buffer.seek(0)
    return audio_buffer

In [5]:
# Define a function to prompt the user to select a text file

def prompt_for_multiple_text_files():
    """Opens a native Windows file explorer to select a text file."""
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True) # Forces window to the front
    print("Opening file selector... Please select one or many text files.")

    # Open the file dialog and allow multiple file selection. The 's' at the 
    #   end of 'askopenfilenames' allows for multiple file selection, 
    #   returning a tuple of file paths. 
    file_paths = filedialog.askopenfilenames(
        title="Select a Text File to Read Aloud",
        filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
    )
    root.destroy()
    return list(file_paths)

In [6]:
# Define the main function to read the selected text file aloud using the TTS function

async def read_text_file_aloud(file_path, rate="-7%", voice="en-US-JennyNeural"):
    """Reads content from the selected text file and plays it chunk by chunk."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = [line.strip() for line in file.readlines() if line.strip()]
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    if not lines:
        print("The selected file is empty.")
        return

    print(f"\nPlaying: {os.path.basename(file_path)}")
    print(f"Voice: {voice} | Speed: {rate}\n")
    
    pygame.mixer.init()
    try:
        for chunk in lines:
            print(f"Reading: {chunk[:50]}...")
            audio_buffer = await text_to_speech(chunk, rate, voice)
            pygame.mixer.music.load(io.BytesIO(audio_buffer.getvalue()))
            pygame.mixer.music.play()
            while pygame.mixer.music.get_busy():
                await asyncio.sleep(0.2)
            await asyncio.sleep(0.4) 
    except Exception as e:
        print(f"Error during playback: {e}")
    finally:
        pygame.mixer.quit()

In [7]:
# Define a function to prompt the user to select a save location for the MP3 file

def prompt_for_save_path(default_filename="audiobook.mp3"):
    """Opens a native Windows dialog to choose where to save the MP3 file."""
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    
    print("Opening save selector... Choose where to save your audio file.")
    
    file_path = filedialog.asksaveasfilename(
        title="Save Audio File As",
        initialfile=default_filename,
        defaultextension=".mp3",
        filetypes=[("MP3 Audio", "*.mp3"), ("All Files", "*.*")]
    )
    
    root.destroy()
    return file_path

In [8]:
# Define the main function to read the text file and save the narration to an MP3 file 

async def save_text_to_mp3(text_file_path, save_audio_path, rate="-7%", voice="en-US-JennyNeural"):
    """Reads a text file and saves the entire synthesized narration to a local MP3 file."""
    try:
        with open(text_file_path, 'r', encoding='utf-8') as file:
            lines = [line.strip() for line in file.readlines() if line.strip()]
    except Exception as e:
        print(f"Error reading text file: {e}")
        return

    if not lines:
        print("The selected text file is empty.")
        return

    print(f"\nCreating: {os.path.basename(save_audio_path)}")
    print(f"Voice: {voice} | Speed: {rate}")
    # # Uncomment if you want a progress message without the tqdm bar
    # print("Processing chapters/lines... Please wait.")  
    
    # DEFINE THE FILENAME UP FRONT FOR PRIVACY AND SCOPING
    text_filename = os.path.basename(text_file_path)

    try:
        # Open the final destination file in Write-Binary ('wb') mode
        with open(save_audio_path, 'wb') as final_mp3:

            # We wrap 'lines' with tqdm() to automatically generate the progress bar
            # 'desc' adds a label to the left of the bar
            for chunk in tqdm(lines, desc="Generating Audiobook"):
                
                # Fetch the audio stream buffer from edge_tts
                audio_buffer = await text_to_speech(chunk, rate, voice)
                
                # Write the raw bytes directly into our local MP3 file
                final_mp3.write(audio_buffer.getvalue())
                
                # ===============================================================
                # NOTE: UNCOMMENT THE LINES BELOW IF WANTING TO TEST DIFFERENT VOICES
                # AND HEAR LIVE AUDIO REPLAY WHILE EXPORTING
                # ===============================================================
                # pygame.mixer.init()
                # pygame.mixer.music.load(io.BytesIO(audio_buffer.getvalue()))
                # pygame.mixer.music.play()
                # while pygame.mixer.music.get_busy():
                #     await asyncio.sleep(0.2)
                # pygame.mixer.quit()
                
                # Tiny rest to be respectful to the Microsoft endpoint servers
                await asyncio.sleep(0.3)

            # # Uncomment to see progress in Processing text snippets in the console 
            # #     instead of using the tqdm progress bar (not recommended since tqdm is 
            # #     more elegant and informative)
            # for chunk in lines:
            #     print(f"Processing text snippet: {chunk[:40]}...")
                
            #     # Fetch the audio stream buffer from edge_tts
            #     audio_buffer = await text_to_speech(chunk, rate, voice)
                
            #     # Write the raw bytes directly into our local MP3 file
            #     final_mp3.write(audio_buffer.getvalue())
                
            #     # ===============================================================
            #     # NOTE: UNCOMMENT THE LINES BELOW IF WANTING TO TEST DIFFERENT VOICES
            #     # AND HEAR LIVE AUDIO REPLAY WHILE EXPORTING
            #     # ===============================================================
            #     # pygame.mixer.init()
            #     # pygame.mixer.music.load(io.BytesIO(audio_buffer.getvalue()))
            #     # pygame.mixer.music.play()
            #     # while pygame.mixer.music.get_busy():
            #     #     await asyncio.sleep(0.2)
            #     # pygame.mixer.quit()
                
            #     # Tiny rest to be respectful to the Microsoft endpoint servers
            #     await asyncio.sleep(0.3)

        print(f'\n🎉 Success! Combined audio for "{text_filename}" saved!')
        
    except Exception as e:
        print(f"Error while writing audio file: {e}")

In [9]:
# Select text files to be loaded
selected_files = prompt_for_multiple_text_files()

if selected_files:
    print(f"Successfully loaded {len(selected_files)} file(s):\n")
    for f in selected_files:
        # Extract just the filename from the full path
        print(f" * {os.path.basename(f)}")
else:
    print("No files selected.")

Opening file selector... Please select one or many text files.
Successfully loaded 4 file(s):

 * A small bird.txt
 * The Cat Who Thought He Was a Dog.txt
 * The Day the Pencils Ran Away.txt
 * The Map in the Attic.txt


In [10]:
# # This cell is for testing the TTS functionality with the selected file. 
# # You can adjust the voice and speed settings as needed.
# # Uncomment the whole cell to run the TTS playback after selecting a file.

# # Quick settings adjustments

# voice_choice = "en-AU-NatashaNeural"
# speed_rate = "-5%"

# if 'selected_files' in locals() and selected_files:
#     # Notice the direct 'await' keyword used here instead of asyncio.run()
#     await read_text_file_aloud(selected_files, rate=speed_rate, voice=voice_choice)
# else:
#     print("Please select a file in Cell 7 first!")

In [11]:
# Configuration
voice_choice = "en-AU-NatashaNeural"
speed_rate = "-5%"

if selected_files:
    # 1. Pick the folder where you want all these files saved
    save_folder = filedialog.askdirectory(title="Select Output Folder")
    
    if save_folder:
        # 2. Loop through every file you selected
        for file_path in selected_files:
            base_name = os.path.basename(file_path)
            file_name_no_ext, _ = os.path.splitext(base_name)
            output_path = os.path.join(save_folder, f"{file_name_no_ext}.mp3")
            
            # 3. Save the current file
            await save_text_to_mp3(file_path, output_path, rate=speed_rate, voice=voice_choice)
    else:
        print("Folder selection cancelled.")
else:
    print("No files selected!")


Creating: A small bird.mp3
Voice: en-AU-NatashaNeural | Speed: -5%


Generating Audiobook:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 Success! Combined audio for "A small bird.txt" saved!

Creating: The Cat Who Thought He Was a Dog.mp3
Voice: en-AU-NatashaNeural | Speed: -5%


Generating Audiobook:   0%|          | 0/10 [00:00<?, ?it/s]


🎉 Success! Combined audio for "The Cat Who Thought He Was a Dog.txt" saved!

Creating: The Day the Pencils Ran Away.mp3
Voice: en-AU-NatashaNeural | Speed: -5%


Generating Audiobook:   0%|          | 0/13 [00:00<?, ?it/s]


🎉 Success! Combined audio for "The Day the Pencils Ran Away.txt" saved!

Creating: The Map in the Attic.mp3
Voice: en-AU-NatashaNeural | Speed: -5%


Generating Audiobook:   0%|          | 0/6 [00:00<?, ?it/s]


🎉 Success! Combined audio for "The Map in the Attic.txt" saved!
